In [9]:
from pathlib import Path
import numpy as np

import rasterio

from whitebox.whitebox_tools import WhiteboxTools
from sklearn.preprocessing import MinMaxScaler

wbt = WhiteboxTools()
wbt.verbose = False

In [3]:
data_dir = Path().resolve()

dem_dir = data_dir / "inference_dem_tiles"
hpmf_dir = data_dir / "inference_hpmf_tiles"

# Create HPMF data folder if it don't exist
hpmf_dir.mkdir(parents=True, exist_ok=True)

In [4]:
def minmax_normalized_image(image):
    # Handle uniform images: if all values are equal, return a zero array to avoid division by zero
    if np.max(image) == np.min(image):
        return np.zeros(image.shape, dtype=np.float32)

    # Replace no data values with ones
    image = np.where(np.isnan(image) | (image == -9999), 1, image)

    scaler = MinMaxScaler()                                               # Initialize MinMaxScaler to scale pixel values between 0 and 1
    flat_normalized_image = scaler.fit_transform(image.reshape(-1, 1))    # Flatten the image for scaler input and apply normalization
    normalized_image = flat_normalized_image.reshape(image.shape)         # Reshape the normalized data back to the original image dimensions

    return normalized_image.astype(np.float32)

In [10]:
# Iterate through all DEM tiles
for dem in dem_dir.iterdir():
    
    # Apply High Pass Median Filter (HPMF) to DEM
    hpmf_file = hpmf_dir / f"{dem.stem}_HPMF.tif"
    wbt.high_pass_median_filter(i=dem, output=hpmf_file, filterx=11, filtery=11)

    with rasterio.open(hpmf_file) as hpmf_raster:
        hpmf_array = hpmf_raster.read(1)       # Read raster values as array
        meta = hpmf_raster.meta.copy()
        
    hpmf_array = minmax_normalized_image(hpmf_array)
    
    with rasterio.open(hpmf_file, "w", **meta) as normalized_hpmf_raster:
        normalized_hpmf_raster.write(hpmf_array.astype(rasterio.float32), 1)
